### Install necessary packages

In [1]:
import sys
!{sys.executable} -m pip install -U langchain langchain-community langchain-ollama faiss-cpu python-dotenv langchain-text-splitters chromadb tiktoken pandas -q


### Import the necessary libraries

In [2]:
import os
import json
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_ollama import OllamaEmbeddings
from langchain_ollama import ChatOllama
from deepeval.models import OllamaModel
from deepeval import evaluate
import pandas as pd
import json, re
from langchain_community.vectorstores import FAISS, Chroma
from langchain_openai import OpenAIEmbeddings
from pathlib import Path
from dotenv import load_dotenv

env_path = Path.cwd().parent / ".env.local"   # eine Ebene höher
print("Trying:", env_path, "exists:", env_path.exists())

load_dotenv(env_path, override=True)

CLOUD_MODEL_BASE_URL = os.getenv("CLOUD_MODEL_BASE_URL")
LOCAL_MODEL_BASE_URL = os.getenv("LOCAL_MODEL_BASE_URL")
OLLAMA_API_KEY = os.getenv("OLLAMA_API_KEY")

Trying: /Users/michelecandolfo/Documents/workspaces/DeepEval/ai-engineering-portfolio/.env.local exists: True


## 1) Create RAG Agent 

In [4]:
from langchain_community.document_loaders import PyPDFLoader, TextLoader


class RAGAgent:
    def __init__(
        self,
        document_paths,
        embedding_model=None,
        chunk_size: int = 500,
        chunk_overlap: int = 50,
        vector_store_class=FAISS,
        k: int = 2, 
    
    ):
        # Allow passing a single string or a list
        if isinstance(document_paths, str):
            document_paths = [document_paths]
        self.document_paths = document_paths
        self.chunk_size = chunk_size
        self.chunk_overlap = chunk_overlap

        
        self.embedding_model = embedding_model or OllamaEmbeddings(
            model="qwen3-embedding:latest",
            base_url="http://localhost:11434"
        )


        self.vector_store_class = vector_store_class
        self.k = k
        self.vector_store = self._load_vector_store()

    def _load_vector_store(self):
        documents = []
        
        for document_path in self.document_paths:
            ext = Path(document_path).suffix.lower()
            if ext == ".pdf":
                loader = PyPDFLoader(document_path)
                docs = loader.load()
            elif ext == ".txt":
                loader = TextLoader(document_path, encoding="utf-8")
                docs = loader.load()
            else:
                raise ValueError(f"Unsupported file format: {ext}")

            splitter = RecursiveCharacterTextSplitter(
                chunk_size=self.chunk_size,
                chunk_overlap=self.chunk_overlap,
                separators=["\n\n", "\n", ". ", "? ", "! ", "; ", ", ", " ", ""],
            )

            documents.extend(splitter.split_documents(docs))

        return self.vector_store_class.from_documents(documents, self.embedding_model)

   
   
    def retrieve(self, query: str) -> list[str]:
        docs = self.vector_store.similarity_search(query, k=self.k)
        context = [doc.page_content for doc in docs]
        
        return context

    def generate(
        self,
        query: str,
        retrieved_docs: list,
        llm_model=None,
        prompt_template: str = None
    ):
        context = "\n".join(retrieved_docs)

        model = llm_model or ChatOllama(
            base_url=CLOUD_MODEL_BASE_URL,
            model="kimi-k2.6:cloud",
            temperature=0.0,
            headers={"Authorization": f"Bearer {OLLAMA_API_KEY}"},
        )

        JSON_FORMAT = """
        {
        "answer": "<a concise, complete answer to the user's query>",
        "citations": [
            "<relevant quoted snippet or summary from source 1>",
            "<relevant quoted snippet or summary from source 2>"
        ]
        }
        """.strip()

        JSON_FALLBACK = """
        {
        "answer": "No relevant information available.",
        "citations": []
        }
        """.strip()

        prompt = prompt_template or f"""
        You are a helpful assistant. Use the context below to answer the user's query.
        Format your response strictly as a JSON object with the following structure:
        {JSON_FORMAT}

        Only include information that appears in the provided context. Do not make anything up.
        Only respond in JSON — no explanations. If nothing relevant is found, respond with:
        {JSON_FALLBACK}

        Context:
        {context}

        Query:
        {query}
        """.strip()

        resp = model.invoke(prompt)
        return resp.content


    def answer(self, query: str):
        retrieved_docs = self.retrieve(query)
        generated_answer = self.generate(query, retrieved_docs)

        try:
            res = json.loads(generated_answer)
            return res, retrieved_docs
        except json.JSONDecodeError:
            return {"error": "Invalid JSON returned from model", "raw_output": generated_answer}

### Test the Rag Agent with a sample query

In [5]:
document_paths = ["theranos_legacy.txt"]
query = "What is the document about? Give me a quick answer."


retriever = RAGAgent(
    document_paths=document_paths
)

answer, retrieved_docs = retriever.answer(query)

print(f'Answer: {answer}')
print(f'Retrieved Docs: {retrieved_docs}')

Answer: {'answer': 'The document describes Theranos’s TheraCloud Health Portal, which automatically uploads NanoDrop 3000 test results for patients and providers to review, trend, and receive AI-powered insights, and lists recent company milestones including FDA approvals, expanded testing, partnerships, and funding.', 'citations': ['All NanoDrop 3000 tests are automatically uploaded to TheraCloud, Theranos’s secure web and mobile platform. Patients and providers can review full diagnostic panels, trend health data over time, and receive personalized insights based on AI-powered analytics.', 'Recent Milestones: - FDA Emergency Use Approval granted for the COVID-19 MicroDrop Panel (2021) - Expanded test menu to include pharmacogenomic testing (Q3 2022) - Strategic licensing deal signed with Medix Korea for Asia-Pacific rollout - Completion of Series F funding round, raising $240M from Fidelity, BlackRock, and Sequoia Capital (Q1 2023)']}
Retrieved Docs: ['TheraCloud™ Health Portal:  \nA

## 2) Create the Dataset

### Create goldens with the synthesizer function based on the retrieval context (document)

In [10]:
import re
from deepeval.synthesizer import Synthesizer
from deepeval.models import OllamaModel, OllamaEmbeddingModel
from deepeval.models.base_model import DeepEvalBaseEmbeddingModel
from deepeval.synthesizer.config import ContextConstructionConfig
from pathlib import Path
from typing import List, Union, Dict, Tuple, Optional
from pydantic import BaseModel
import os


class SafeOllamaEmbedder(DeepEvalBaseEmbeddingModel):
    """Embedder that truncates long texts to avoid context length errors."""
    MAX_CHARS = 8000  # qwen3-embedding has 40K context, much more generous

    def __init__(self, model="qwen3-embedding:latest", host="http://localhost:11434"):
        self._inner = OllamaEmbeddingModel(model=model, host=host)
        self._model_name = model

    def embed_text(self, text: str) -> List[float]:
        return self._inner.embed_text(text[:self.MAX_CHARS])

    def embed_texts(self, texts: List[str]) -> List[List[float]]:
        return [self.embed_text(t) for t in texts]

    async def a_embed_text(self, text: str) -> List[float]:
        return await self._inner.a_embed_text(text[:self.MAX_CHARS])

    async def a_embed_texts(self, texts: List[str]) -> List[List[float]]:
        results = []
        for text in texts:
            result = await self.a_embed_text(text)
            results.append(result)
        return results

    def load_model(self, async_mode: bool = False):
        return self._inner.load_model(async_mode=async_mode)

    def get_model_name(self):
        return f"{self._model_name} (Safe)"


class PatchedOllamaModel(OllamaModel):
    """OllamaModel that strips markdown code blocks from JSON responses."""

    @staticmethod
    def _strip_markdown_json(text: str) -> str:
        text = text.strip()
        # Remove ```json ... ``` or ``` ... ``` wrappers
        match = re.search(r'```(?:json)?\s*(\{.*?\})\s*```', text, re.DOTALL)
        if match:
            return match.group(1)
        match = re.search(r'```(?:json)?\s*(\[.*?\])\s*```', text, re.DOTALL)
        if match:
            return match.group(1)
        # Fallback: find first { ... } or [ ... ]
        match = re.search(r'\{.*\}', text, re.DOTALL)
        if match:
            return match.group(0)
        match = re.search(r'\[.*\]', text, re.DOTALL)
        if match:
            return match.group(0)
        return text

    def generate(
        self, prompt: str, schema: Optional[BaseModel] = None
    ) -> Tuple[Union[str, Dict], float]:
        chat_model = self.load_model()
        response = chat_model.chat(
            model=self.model_name,
            messages=[{"role": "user", "content": prompt}],
            format=schema.model_json_schema() if schema else None,
            options={
                **{"temperature": self.temperature},
                **self.generation_kwargs,
            },
        )
        content = response.message.content
        if schema:
            content = self._strip_markdown_json(content)
            return schema.model_validate_json(content), 0
        return content, 0


judgeModel = PatchedOllamaModel(
    model="qwen3-next:80b-cloud",
    base_url=CLOUD_MODEL_BASE_URL,
    temperature=0.0,
    headers={"Authorization": f"Bearer {OLLAMA_API_KEY}"},
)

ollamaEmbedder = SafeOllamaEmbedder(model="qwen3-embedding:latest", host="http://localhost:11434")

config = ContextConstructionConfig(
    embedder=ollamaEmbedder,
    critic_model=judgeModel,
    chunk_size=256,
    chunk_overlap=50,
)

synthesizer = Synthesizer(model=judgeModel, async_mode=False)

pdf_path = str(Path("ISO_25059-2026-01.pdf").resolve())
print(f"PDF path: {pdf_path}")
print(f"File exists: {Path(pdf_path).exists()}")

goldens = synthesizer.generate_goldens_from_docs(
    document_paths=[pdf_path],
    context_construction_config=config,
)

print(f"Generated {len(goldens)} goldens")

Output()

PDF path: /Users/michelecandolfo/Documents/workspaces/DeepEval/ai-engineering-portfolio/02_The_RAG_Agent/ISO_25059-2026-01.pdf
File exists: True


[Confident AI Synthesizer Log] SUCCESS: Successfully deleted: 
/var/folders/3f/_f4mzqvd4hxfxdm0glfcz4z80000gp/T/deepeval_chroma_zjvj972g

KeyboardInterrupt: 

### Push dataset into ConfidentAI

In [8]:
from deepeval.dataset import EvaluationDataset

dataset = EvaluationDataset(goldens=goldens)
dataset.push(alias="Vorlesungs Dataset 2")

✅ Dataset successfully pushed to Confident AI! View at 
]8;id=738640;https://app.confident-ai.com/project/cme1cbz3902nx419rccdjpuxv/datasets/cmpcf83vj0009nw13fmhhx2lx\https://app.confident-ai.com/project/cme1cbz3902nx419rccdjpuxv/datasets/cmpcf83vj0009nw13fmhhx2lx]8;;\

## 3) Evaluate the RAG Agent based on specific metrics

### Pull the created dataset from ConfidentAI

In [6]:
from deepeval.test_case import LLMTestCase
from deepeval.dataset import EvaluationDataset


dataset = EvaluationDataset()
dataset.pull("Vorlesungs Dataset 2")

Output()

### Create the RAG Agent

In [7]:
"""
Standard config:

embedding_model=nomic-embed-text:latest,
chunk_size: int = 500,
chunk_overlap: int = 50,
vector_store_class=FAISS,
k: int = 2, 

"""

agent = RAGAgent("ISO_25059-2026-01.pdf")

### Generate LLMTestCases from Goldens with the candidate models answer

In [8]:
test_cases = []
for golden in dataset.goldens:
    retrieved_docs = agent.retrieve(golden.input)
    response = agent.generate(golden.input, retrieved_docs)
    test_case = LLMTestCase(
        input=golden.input,
        actual_output=str(response),
        retrieval_context=retrieved_docs,
        expected_output=golden.expected_output
    )
    test_cases.append(test_case)

print(f'Amount of LLMTestCases generated: {len(test_cases)}')


Amount of LLMTestCases generated: 6


### Create the Judge LLM to evaluate the generated answers based on specific metrics

In [11]:
import re
from deepeval.models import OllamaModel

class CleanOllamaModel(OllamaModel):
    """OllamaModel that strips markdown code blocks from LLM responses."""
    
    @staticmethod
    def _strip_markdown_json(text: str) -> str:
        return re.sub(r'^```(?:json)?\s*\n?', '', text.strip()).rstrip('`').strip()

    def generate(self, prompt, schema=None):
        result, cost = super().generate(prompt, schema)
        if isinstance(result, str):
            result = self._strip_markdown_json(result)
            if schema:
                result = schema.model_validate_json(result)
        return result, cost

    async def a_generate(self, prompt, schema=None):
        result, cost = await super().a_generate(prompt, schema)
        if isinstance(result, str):
            result = self._strip_markdown_json(result)
            if schema:
                result = schema.model_validate_json(result)
        return result, cost

In [12]:
# PatchedOllamaModel (aus Zelle oben) statt normalem OllamaModel
judgeModel = CleanOllamaModel(
    model="qwen3-next:80b-cloud",
    base_url=CLOUD_MODEL_BASE_URL,
    temperature=0.0,
    headers={"Authorization": f"Bearer {OLLAMA_API_KEY}"},
)

### Create specific metrics to evaluate the generated answers

In [13]:
from deepeval.metrics import (
    ContextualRelevancyMetric,
    ContextualRecallMetric,
    ContextualPrecisionMetric,
    GEval
)

from deepeval.test_case import LLMTestCaseParams
from deepeval import evaluate

#! Retriever Metrics

relevancy = ContextualRelevancyMetric(model=judgeModel)
recall = ContextualRecallMetric(model=judgeModel)
precision = ContextualPrecisionMetric(model=judgeModel)


#! Generator Metrics

answer_correctness = GEval(
    name="Answer Correctness",
    model=judgeModel,
    criteria="Evaluate if the actual output's 'answer' property is correct and complete from the input and retrieved context. If the answer is not correct or complete, reduce score.",
    evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT, LLMTestCaseParams.RETRIEVAL_CONTEXT]
)

citation_accuracy = GEval(
    name="Citation Accuracy",
    model=judgeModel,
    criteria="Check if the citations in the actual output are correct and relevant based on input and retrieved context. If they're not correct, reduce score.",
    evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT, LLMTestCaseParams.RETRIEVAL_CONTEXT]
)


#! Types of metrics:

retriever_metrics = [relevancy, recall, precision]
generator_metrics = [answer_correctness, citation_accuracy]

### Evaluate the generated answers based on the specific metrics and create a report

In [16]:
from deepeval.evaluate import AsyncConfig

res_retriver = evaluate(test_cases=test_cases, metrics=retriever_metrics, async_config=AsyncConfig(max_concurrent=3))
test_results = getattr(res_retriver, "test_results", res_retriver)

✨ You're running DeepEval's latest Contextual Relevancy Metric! (using qwen3-next:80b-cloud (Ollama), 
strict=False, async_mode=True)...

✨ You're running DeepEval's latest Contextual Recall Metric! (using qwen3-next:80b-cloud (Ollama), strict=False, 
async_mode=True)...

✨ You're running DeepEval's latest Contextual Precision Metric! (using qwen3-next:80b-cloud (Ollama), 
strict=False, async_mode=True)...

Output()



Metrics Summary

  - ✅ Contextual Relevancy (score: 0.5, threshold: 0.5, strict: False, evaluation model: qwen3-next:80b-cloud (Ollama), reason: The score is 0.50 because while the statements about 'system specific consequences that cannot be interpreted using a straight-line linear scale' and 'adapting itself to a changing dynamic environment' are relevant, the context also includes mentions of continuous learning (e.g., 'AI systems can learn from new training data, production data and the results of previous actions taken by the system') and model switching which are explicitly excluded per the input., error: None)
  - ❌ Contextual Recall (score: 0.2857142857142857, threshold: 0.5, strict: False, evaluation model: qwen3-next:80b-cloud (Ollama), reason: The score is 0.29 because only the first sentence and conclusion are partially supported by nodes in retrieval context (nodes 2 and 1), while all five detailed points lack support., error: None)
  - ✅ Contextual Precision (score: 1.0

⚠ WARNING: No hyperparameters logged.
» ]8;id=393069;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Done 🎉! View results on 
]8;id=927381;https://app.confident-ai.com/project/cme1cbz3902nx419rccdjpuxv/test-runs/cmpd11i090001pb13yfqj43hl/test-cases\https://app.confident-ai.com/project/cme1cbz3902nx419rccdjpuxv/test-runs/cmpd11i090001pb13yfqj43hl/test-cases]8;;\

In [ ]:
from deepeval.evaluate import AsyncConfig

res_generator = evaluate(test_cases=test_cases, metrics=generator_metrics, async_config=AsyncConfig(run_async=False))
test_results = getattr(res_generator, "test_results", res_generator)

## 4) Improving the Retriever

In [ ]:

dataset = EvaluationDataset()
dataset.pull("Vorlesungs Dataset")

relevancy = ContextualRelevancyMetric(model=judgeModel)
recall = ContextualRecallMetric(model=judgeModel)
precision = ContextualPrecisionMetric(model=judgeModel)

metrics = [relevancy, recall, precision]

#chunk_sizes = [512]
#chunk_overlaps = [0]
#ks = [2]

chunk_sizes = [256, 384]
ks = [2, 3]

embedding_models = [
    #("Ollama_nomic_embed_text", OllamaEmbeddings(model="nomic-embed-text:latest", base_url=OLLAMA_BASE_URL,)),
    ("OpenAIEmbeddings", OpenAIEmbeddings()),
    #("Ollama_embeddinggemma", OllamaEmbeddings(model="embeddinggemma:latest", base_url=OLLAMA_BASE_URL,)),
]
vector_store_classes = [
    ("FAISS", FAISS),
    #("Chroma", Chroma)
]

document_paths = ["theranos_legacy.txt"]

results_log = []

for chunk_size in chunk_sizes:
    for k in ks:
        for embedding_name, embedding_model in embedding_models:
            for store_name, store_cls in vector_store_classes:

                retriever = RAGAgent(
                    document_paths=document_paths,
                    embedding_model=embedding_model,
                    chunk_size=chunk_size,
                    vector_store_class=store_cls,
                    k=k,
                )

                retriever_test_cases = []
                for golden in dataset.goldens:
                    retrieved_docs = retriever.retrieve(golden.input)

                    retriever_test_cases.append(
                        LLMTestCase(
                            input=golden.input,
                            actual_output="",  # Dummy für retrieval-only
                            expected_output=golden.expected_output,
                            retrieval_context=retrieved_docs,
                        )
                    )

                run_id = (
                    f"store={store_name} | chunk={chunk_size}"
                    f"| emb={embedding_name} | k={k}"
                )

                

                result = evaluate(
                    retriever_test_cases,
                    metrics,
                    identifier=run_id,
                    hyperparameters={
                        "chunk_size": chunk_size,
                        "embedding_name": embedding_name,
                        "vector_store_class": store_name,
                        "k": k,
                        "doc": "theranos_legacy.txt",
                    },
                )

                results_log.append({
                "run_id": run_id,
                "store": store_name,
                "chunk_size": chunk_size,
                "k": k,
                "embedding": embedding_name,
                "result": result,  # erstmal raw speichern; später extrahieren
                })

Output()

✨ You're running DeepEval's latest Contextual Relevancy Metric! (using gpt-oss:latest (Ollama), strict=False, 
async_mode=True)...

✨ You're running DeepEval's latest Contextual Recall Metric! (using gpt-oss:latest (Ollama), strict=False, 
async_mode=True)...

✨ You're running DeepEval's latest Contextual Precision Metric! (using gpt-oss:latest (Ollama), strict=False, 
async_mode=True)...

Output()

...

✨ You're running DeepEval's latest Contextual Precision Metric! (using gpt-oss:latest (Ollama), strict=False, 
async_mode=True)...

Output()



Metrics Summary

  - ✅ Contextual Relevancy (score: 0.7142857142857143, threshold: 0.5, strict: False, evaluation model: gpt-oss:latest (Ollama), reason: The score is 0.71 because the retrieval context includes useful details such as "The NanoDrop 3000 is a compact, portable diagnostic device capable of performing over 300 blood tests using just 1–2 microliters of capillary blood," which directly relates to ambulance diagnostics, but also contains unrelated information like "The statement discusses a home kit and telehealth coverage, which has nothing to do with the use of NanoDrop 3000 in ambulances or its impact on emergency diagnostics and triage," which lowers overall relevance., error: None)
  - ✅ Contextual Recall (score: 0.5, threshold: 0.5, strict: False, evaluation model: gpt-oss:latest (Ollama), reason: The score is 0.50 because sentences 2 and 4 of the expected output are supported by node 1 and node 2 in the retrieval context, while sentences 1 and 3 lack corresponding no

⚠ WARNING: No prompts logged.
» ]8;id=122070;https://deepeval.com/docs/evaluation-prompts\Log prompts]8;;\ to evaluate and optimize your prompt templates and models.

================================================================================

✓ Done 🎉! View results on 
]8;id=529559;https://app.confident-ai.com/project/cme1cbz3902nx419rccdjpuxv/test-runs/cmlnvswvg02jolg1eiuo1x74k/test-cases\https://app.confident-ai.com/project/cme1cbz3902nx419rccdjpuxv/test-runs/cmlnvswvg02jolg1eiuo1x74k/test-cases]8;;\

In [ ]:
from deepeval.dataset import EvaluationDataset
from deepeval.test_case import LLMTestCase, LLMTestCaseParams
from langchain_ollama import ChatOllama
from deepeval import evaluate


from langchain_openai import OpenAIEmbeddings

from langchain_community.vectorstores import Chroma


dataset = EvaluationDataset()
dataset.pull("RAG QA Agent Dataset")

metrics = [answer_correctness, citation_accuracy] # i defined the metrics before

document_paths = ["theranos_legacy.txt"]

models = [
    ("gpt-oss", ChatOllama(
        base_url=OLLAMA_BASE_URL,
        model="qwen3:latest",
        temperature=0.0
    )),
]

for model_name, model in models:
    retriever = RAGAgent(
        document_paths,
        embedding_model=OpenAIEmbeddings(),
        chunk_size=1024,
        vector_store_class=Chroma,
    ) # Initialize retriever with new configuration

    generator_test_cases = []
    for golden in dataset.goldens:
        retrieved_docs = retriever.retrieve(golden.input)     # list[str]
        answer_json_str = retriever.generate(golden.input, retrieved_docs, llm_model=model)
        context_list = retrieved_docs  # already list[str]
        test_case = LLMTestCase(
            input=golden.input,
            actual_output=answer_json_str,
            retrieval_context=context_list
        )
        generator_test_cases.append(test_case)

    run_id = f"{model_name}"

    evaluate(
        generator_test_cases,
        metrics,
        identifier=run_id,
        hyperparameters={
            "model": model_name,
            "doc": "theranos_legacy.txt",
        }
    )